In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
HERE = %pwd
sys.path.append(os.path.dirname(HERE))

%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
    
import numpy as np
import pandas as pd
import copy
import pickle
import time
import collections
from tqdm import tqdm

In [2]:
from src import utils
rng = utils.set_seed()

dir_parent = utils.dir_parent
version_exp = utils.version_exp
dir_workspace = f"{dir_parent}/research/TFCSR"

In [3]:
dir_load = f"{dir_parent}/received_data/job-recommendation"
dir_data = f"{dir_workspace}/preprocessed_data/{version_exp}/Job"
os.makedirs(dir_data, exist_ok=True)

# parameters
n_positive = 2
n_candidate = 50 - n_positive
n_user = 500

# Load data

In [4]:
# item master
df_items = pd.read_csv(f'{dir_load}/jobs.tsv', delimiter='\t', encoding='utf-8', on_bad_lines='skip').fillna("")
df_items = df_items.drop(["WindowID", "Zip5"], axis=1)
df_items = df_items.rename(columns={"JobID" : "itemID"}).set_index("itemID")
df_items.columns = [utils.change_col(a) for a in df_items.columns]
print(f"#jobs (original) : {len(df_items)}")
df_items.head()

/tmp/ipykernel_3544468/196657431.py:2: DtypeWarning: Columns (0: Zip5) have mixed types. Specify dtype option on import or set low_memory=False.
  df_items = pd.read_csv(f'{dir_load}/jobs.tsv', delimiter='\t', encoding='utf-8', on_bad_lines='skip').fillna("")


#jobs (original) : 1091923


,title,description,requirements,city,state,country,start date,end date
itemID,,,,,,,,
1,Security Engineer/Technical Lead,<p>Security Clearance Required:&nbsp; Top Secr...,<p>SKILL SET</p>\r<p>&nbsp;</p>\r<p>Network Se...,Washington,DC,US,2012-03-07 13:17:01.643,2012-04-06 23:59:59
4,SAP Business Analyst / WM,<strong>NO Corp. to Corp resumes&nbsp;are bein...,<p><b>WHAT YOU NEED: </b></p>\r<p>Four year co...,Charlotte,NC,US,2012-03-21 02:03:44.137,2012-04-20 23:59:59
7,P/T HUMAN RESOURCES ASSISTANT,<b> <b> P/T HUMAN RESOURCES ASSISTANT</b> <...,Please refer to the Job Description to view th...,Winter Park,FL,US,2012-03-02 16:36:55.447,2012-04-01 23:59:59
8,Route Delivery Drivers,CITY BEVERAGES Come to work for the best in th...,Please refer to the Job Description to view th...,Orlando,FL,US,2012-03-03 09:01:10.077,2012-04-02 23:59:59
9,Housekeeping,I make sure every part of their day is magica...,Please refer to the Job Description to view th...,Orlando,FL,US,2012-03-03 09:01:11.88,2012-04-02 23:59:59


In [5]:
%%time
df_items['description'] = df_items['description'].apply(utils.to_text)
df_items['requirements'] = df_items['requirements'].apply(utils.to_text)

CPU times: user 41.4 s, sys: 1.82 s, total: 43.3 s
Wall time: 43.4 s


In [6]:
# reduce jobs whose description and requirements are long
def _fn(s):
    try:
        n = utils.compute_token(s)
    except:
        n = 0
    return n

s1 = np.array([_fn(s) for s in tqdm(df_items["description"].values)])
s2 = np.array([_fn(s) for s in tqdm(df_items["requirements"].values)])
df_items = df_items[(s1 < 1000) * (s1 > 10) * (s2 < 500) * (s2 > 10)]
print(f"#jobs (reduced) : {len(df_items)}")
df_items = df_items.T.add_prefix("I").T
df_items.head()

100%|████████████████████████████████████████████████████████████████████████████████████| 1091923/1091923 [01:51<00:00, 9770.76it/s]


#jobs (reduced) : 781020


,title,description,requirements,city,state,country,start date,end date
itemID,,,,,,,,
I1,Security Engineer/Technical Lead,Security Clearance Required: Top Secret\nJob ...,SKILL SET\nNetwork Security tools:\nWebdefend ...,Washington,DC,US,2012-03-07 13:17:01.643,2012-04-06 23:59:59
I4,SAP Business Analyst / WM,NO Corp. to Corp resumes are being considered ...,WHAT YOU NEED:\nFour year college degree\nMini...,Charlotte,NC,US,2012-03-21 02:03:44.137,2012-04-20 23:59:59
I7,P/T HUMAN RESOURCES ASSISTANT,P/T HUMAN RESOURCES ASSISTANT\n—— 1-2 years e...,Please refer to the Job Description to view th...,Winter Park,FL,US,2012-03-02 16:36:55.447,2012-04-01 23:59:59
I8,Route Delivery Drivers,CITY BEVERAGES Come to work for the best in th...,Please refer to the Job Description to view th...,Orlando,FL,US,2012-03-03 09:01:10.077,2012-04-02 23:59:59
I9,Housekeeping,I make sure every part of their day is magica...,Please refer to the Job Description to view th...,Orlando,FL,US,2012-03-03 09:01:11.88,2012-04-02 23:59:59


In [7]:
# user master
df_users = pd.read_csv(f'{dir_load}/users.tsv', sep='\t').fillna("")

def _fn(s):
    try:
        year = int(s.split("-")[0])
    except:
        year = ""
    return year


df_users["GraduationDate"] = df_users["GraduationDate"].apply(_fn)
df_users = df_users.drop(["City", "State", "Country", "WindowID", "Split", "ZipCode"], axis=1)
df_users = df_users.rename(columns={"UserID" : "userID", "GraduationDate" : "GraduationYear"}).set_index("userID").T.add_prefix("U").T

def _tmp(s):
    try:
        s = int(s)
    except:
        s = ""
    return s
df_users["TotalYearsExperience"] = [_tmp(s) for s in tqdm(df_users["TotalYearsExperience"].values)]
df_users = df_users[(df_users["Major"] != "") * (df_users["Major"] != "Not Applicable")]
df_users.columns = [utils.change_col(a) for a in df_users.columns]
print(f"#users : {len(df_users)}")
df_users.head()

100%|███████████████████████████████████████████████████████████████████████████████████| 389708/389708 [00:00<00:00, 4320787.27it/s]


#users : 233645


,degree type,major,graduation year,work history count,total years experience,currently employed,managed others,managed how many
userID,,,,,,,,
U72,Master's,Anthropology,2011,10,8,Yes,No,0
U98,Master's,Journalism,2007,3,3,Yes,No,0
U123,Bachelor's,Agricultural Business,2011,1,9,Yes,No,0
U131,Bachelor's,Finance,1998,3,14,,No,0
U162,Master's,I/O Psychology,2012,10,25,No,No,0


In [8]:
# transaction records (job application logs)
df_records = pd.read_csv(f'{dir_load}/apps.tsv', delimiter='\t', encoding='utf-8').fillna("")
df_records = df_records.drop(["WindowID", "Split"], axis=1)
df_records = df_records.rename(columns={"UserID" : "userID", "JobID" : "itemID"})
df_records["itemID"] = df_records["itemID"].apply(lambda s : f"I{s}")
df_records["userID"] = df_records["userID"].apply(lambda s : f"U{s}")
print(len(df_records))

# reduce invalid item IDs
items = df_items.index.values
gb = df_records.groupby("itemID")
d_ = dict()
for item in tqdm(items):
    try:
        d_[item] = gb.get_group(item)
    except:
        pass
df_records = pd.concat(d_.values())
print(len(df_records))
df_records.head()

1603111


100%|█████████████████████████████████████████████████████████████████████████████████████| 781020/781020 [00:34<00:00, 22746.17it/s]


1312867


,userID,ApplicationDate,itemID
20456,U117932,2012-04-03 11:59:11.767,I1
184774,U853630,2012-04-05 17:56:52.017,I1
350426,U1457715,2012-04-06 17:54:32.95,I1
79860,U367854,2012-04-04 14:59:25.78,I10
219726,U981444,2012-04-02 10:25:00.923,I10


In [9]:
# job history (!= job application logs)
df_logs = pd.read_csv(f'{dir_load}/user_history.tsv', delimiter='\t', encoding='utf-8')
df_logs = df_logs.drop(["WindowID", "Split"], axis=1).dropna()
df_logs = df_logs.rename(columns={"UserID" : "userID"})
df_logs["userID"] = df_logs["userID"].apply(lambda s : f"U{s}")

users = df_logs["userID"].unique()
gb = df_logs.groupby("userID")

d_logs = dict()
for user in tqdm(users):
    df_ = gb.get_group(user).sort_values(by="Sequence", ascending=True)
    if len(df_) > 1 and len(df_) < 10:
        d_logs[user] = {f"{i+1}" : t for i,t in enumerate(df_["JobTitle"].values)}
pd.DataFrame(d_logs).T.head()

100%|██████████████████████████████████████████████████████████████████████████████████████| 371869/371869 [00:57<00:00, 6450.09it/s]


,1,2,3,4,5,6,7,8,9
U47,National Space Communication Programs-Special ...,Detention Officer,"Passenger Screener, TSA",NaN,NaN,NaN,NaN,NaN,NaN
U72,"Lecturer, Department of Anthropology",Student Assistant,Elderly Caregiver,"Department Assistant, Department",Book Reviews Editorial Assistant,Graduate Assistant,Research Assistant,"Docent, C.E. Smith Museum of Anthropology, Cal...",NaN
U80,"Auto Publishing/Electro Mechanical Technician,...","Enhanced Baker Cell Technician, EBC Technician",Lead was was also given the position as Drug ...,"Sales Associate, Installer","Enhanced Baker Cell Technician, EBC Technician",NaN,NaN,NaN,NaN
U98,Editor-in-Chief,Deputy Sports & Website Editor,Author,NaN,NaN,NaN,NaN,NaN,NaN
U131,Data Analyst,Assistant Business Analyst,Financial Form Specialist,NaN,NaN,NaN,NaN,NaN,NaN


# Create preprocessed data

In [10]:
# frequent items up to rank 10000
s = df_records["itemID"].value_counts()
s = s.sort_values(ascending=False).iloc[:1000]
set_items = set(s.index)

gb = df_records.groupby("userID")
users = df_records["userID"].unique()
k = -1*n_positive


def _items_user(user, gb):
    df_ = gb.get_group(user).sort_values(by="ApplicationDate", ascending=True)
    items_user = df_["itemID"].values
    return items_user

def _candidate_items(items_user, set_items, k, n_candidate, df_items):
    items_pos = items_user[k:]  # latest k items
    
    ## jobs that have the same location as the positive job
    v = df_items.loc[items_pos][["state", "country"]].values
    l = list(dict.fromkeys(tuple(row) for row in v))
    locations_true = np.array(l, dtype=object)
    
    d_ = dict()
    for i, location in enumerate(locations_true):
        s = (df_items[["state", "country"]] == location)
        d_[i] = df_items[s.sum(axis=1) == len(location)]
    
    df_items_same_location = pd.concat(d_.values()).drop_duplicates()
    
    ## select negative items from the same location jobs
    set_i = set(df_items_same_location.index.values)
    set_i = set_i - set(items_pos)
    items_ = np.array(list(set_items.intersection(set_i)))
    if len(items_) >= n_candidate:
        items_neg_same_location = rng.choice(items_, size=n_candidate, replace=False)
    else:
        items_neg_same_location = items_
    
    n_ = len(items_neg_same_location)
    if n_candidate > n_:
        ## select negative items from the other location jobs
        items_neg_other_location = rng.choice(list(set_items - set(items_neg_same_location)), size=n_candidate - n_, replace=False)
        items_neg = np.concatenate([items_neg_same_location, items_neg_other_location])
    else:
        items_neg = items_neg_same_location
    
    d_ = {
        "candidates_positive" : ", ".join(items_pos),
        "candidates_negative" : ", ".join(items_neg)
    }
    return copy.deepcopy(d_)

dd_data = dict()

### User profile

In [11]:
for user_type, flag in zip(["mid-career", "new-graduate"], [True, False]):
    # shuffle
    users = rng.choice(users, size=len(users), replace=False)
    
    d_data = dict()
    idx = 0
    for user in tqdm(users):
        items_user = _items_user(user, gb)
        
        if len(items_user) >= n_positive:
            try:
                # profile
                dp = df_users.loc[user].to_dict()
                try:
                    dp["work history"] = d_logs[user]
                    flag_work_history = True
                except:
                    flag_work_history = False     
        
                if flag_work_history == flag:
                    # candidate items
                    d_ = _candidate_items(items_user, set_items, k, n_candidate, df_items)
                    d_["profile"] = dp
                    d_data[user] = copy.deepcopy(d_)
                    idx += 1
            except:
                pass
        
        if idx == n_user:
            print(user, d_data[user])
            break

    dd_data[f"profile_{user_type}"] = copy.deepcopy(d_data)

  1%|▍                                                                                      | 1491/293967 [03:41<12:02:44,  6.74it/s]


U257063 {'candidates_positive': 'I403897, I473393', 'candidates_negative': 'I342631, I1076994, I916686, I300053, I520868, I332636, I220696, I216529, I456571, I600704, I820934, I900470, I681911, I1104406, I824093, I1059686, I900722, I520678, I300712, I867261, I900397, I596875, I178337, I283201, I820820, I600212, I1051365, I261263, I574972, I1104352, I900368, I635, I521315, I584169, I142, I600506, I820929, I900355, I862393, I300118, I282616, I300527, I771740, I212033, I789255, I789256, I863649, I488549', 'profile': {'degree type': "Bachelor's", 'major': 'Criminal Justice/ Management', 'graduation year': 2013, 'work history count': 3, 'total years experience': 19, 'currently employed': 'Yes', 'managed others': 'No', 'managed how many': 0, 'work history': {'1': 'Sales Associate / specialist', '2': 'Co-Manager and Chef Chef', '3': 'Manager/ Owner'}}}


  4%|███▋                                                                                   | 12418/293967 [03:24<1:17:11, 60.78it/s]

U1268007 {'candidates_positive': 'I646717, I1004596', 'candidates_negative': 'I196188, I188362, I295335, I846871, I741664, I639131, I617448, I427, I637250, I755177, I686211, I851436, I900299, I130816, I846882, I158317, I997250, I1107108, I530713, I458151, I378442, I130364, I844927, I212574, I273258, I176737, I209532, I1054840, I246613, I911713, I563184, I1107344, I218261, I574359, I38706, I1056554, I812431, I699316, I846883, I959948, I228642, I768139, I900295, I98665, I216242, I246235, I846134, I1035292', 'profile': {'degree type': '', 'major': 'Business Administration', 'graduation year': 2006, 'work history count': 4, 'total years experience': 4, 'currently employed': 'No', 'managed others': 'No', 'managed how many': 0}}


### User history

In [12]:
N_icl = [1, 3, 5, 10]
for n_icl in N_icl:
    # shuffle
    users = rng.choice(users, size=len(users), replace=False)
    
    d_data = dict()
    idx = 0
    for user in tqdm(users):
        items_user = _items_user(user, gb)

        if len(items_user) >= n_icl + n_positive:
            try:
                # candidate items
                d_ = _candidate_items(items_user, set_items, k, n_candidate, df_items)
                
                # user history (except latest k items)
                d_["history"] = {i+1 : item for i,item in enumerate(items_user[:k][::-1][:n_icl])} 
                d_data[user] = copy.deepcopy(d_)
                idx += 1
            except:
                pass
            
        if idx == n_user:
            print(user, d_data[user])
            break

    dd_data[f"{n_icl}-sample"] = copy.deepcopy(d_data)

  0%|▎                                                                                      | 1210/293967 [03:22<13:37:47,  5.97it/s]


U34251 {'candidates_positive': 'I476393, I1053259', 'candidates_negative': 'I681377, I863441, I522604, I922828, I141861, I922917, I959570, I312203, I631466, I31688, I494873, I545906, I331259, I631311, I1055024, I297931, I144635, I1052995, I31575, I31744, I204917, I430387, I922943, I922981, I192035, I1070792, I31496, I200305, I563582, I922816, I696309, I478877, I627377, I152385, I879949, I614, I975389, I680446, I342749, I600366, I220735, I634722, I811103, I13452, I1062654, I997250, I901317, I517780', 'history': {1: 'I379054'}}


  1%|▌                                                                                       | 2075/293967 [03:19<7:48:39, 10.38it/s]


U497948 {'candidates_positive': 'I566294, I495138', 'candidates_negative': 'I1038347, I996970, I212988, I1076104, I1115793, I182116, I842289, I1102992, I882072, I695188, I194603, I281243, I476620, I899195, I242605, I358314, I741491, I852577, I1099683, I182113, I879949, I613, I1035339, I600649, I1014434, I200288, I441773, I752580, I61815, I951860, I379547, I91846, I819544, I244778, I302922, I1041406, I474308, I148899, I482488, I83584, I410253, I242710, I794498, I997738, I284579, I134765, I899777, I843815', 'history': {1: 'I867893', 2: 'I928004', 3: 'I541652'}}


  1%|▉                                                                                       | 3139/293967 [03:22<5:13:08, 15.48it/s]


U1355857 {'candidates_positive': 'I911256, I625834', 'candidates_negative': 'I852823, I700868, I576753, I576752, I700867, I489604, I282146, I375589, I330378, I573180, I709999, I812299, I330348, I587248, I651189, I768508, I330380, I337015, I391770, I908493, I899694, I461834, I928713, I500810, I543622, I261539, I790782, I188362, I584169, I908559, I608556, I143792, I1012931, I815857, I331259, I362519, I456571, I158490, I434528, I403898, I602498, I638812, I152528, I595723, I600613, I1074471, I577304, I511573', 'history': {1: 'I330382', 2: 'I489604', 3: 'I957686', 4: 'I936513', 5: 'I922128'}}


  2%|██                                                                                      | 6716/293967 [03:16<2:20:16, 34.13it/s]

U840906 {'candidates_positive': 'I811927, I782692', 'candidates_negative': 'I788041, I261441, I577029, I314766, I212093, I812531, I780646, I596389, I282316, I1053892, I335155, I1001363, I837446, I300809, I300660, I198213, I109062, I782712, I483046, I1050985, I282642, I940900, I272504, I874549, I338167, I681893, I150521, I680446, I23302, I896947, I512814, I550180, I386591, I871285, I752659, I27752, I158490, I35069, I300974, I314425, I638811, I338734, I281073, I674546, I749238, I773030, I811946, I362518', 'history': {1: 'I149354', 2: 'I1032353', 3: 'I638479', 4: 'I889499', 5: 'I601021', 6: 'I584241', 7: 'I1045358', 8: 'I512686', 9: 'I812531', 10: 'I478877'}}


### Save

In [13]:
d_item = dict()
for user_type, d_data in dd_data.items():
    with open(f"{dir_data}/records_{user_type}.pickle", 'wb') as f:
        pickle.dump(d_data, f)
    
    # items that used in the main experiments
    try:
        items_history = np.unique(np.concatenate([list(d["history"].values()) for d in d_data.values()]))
    except:
        items_history = []
    
    items_pos = np.unique(np.concatenate([d["candidates_positive"].split(", ") for d in d_data.values()]))
    items_neg = np.unique(np.concatenate([d["candidates_negative"].split(", ") for d in d_data.values()]))
    items_candidates = np.unique(np.concatenate([items_pos, items_neg]))
            
    d_item[user_type] = {
        "history" : items_history,
        "candidates" : items_candidates
    }

for item_type in ["history", "candidates"]:
    items_ = np.unique(np.concatenate([d[item_type] for d in d_item.values()]))
    df_i = df_items.loc[items_]
    df_i = df_i[['title', 'description', 'requirements']]
    df_i.to_csv(f"{dir_data}/items_{item_type}.csv")
    display(df_i.head())
    print(f"{len(df_i)} items for {item_type} record.")

,title,description,requirements
itemID,,,
I1000011,Front Desk Recruiter/Receptionist,Front Desk Recruiter/Receptionist\nRapidly Gro...,Valid DL\nHigh School Diploma\nAbility to Mult...
I1000056,Material Handler,"Our client, a well-known health care solutions...",Qualifications:\nBasic computer skills\nExperi...
I1000063,Office Assistant,Our client is a professional health care organ...,"Qualifications:\nMicrosoft Word, Excel and Out..."
I1000690,Deployment Project Manager - Telecommunications,Deployment Project Manager - Telecommunication...,Deployment Project Manager - Telecommunication...
I100096,Inventory Manager,Ingersoll Rand is uniquely qualified to create...,• Problem Solving: Balances inventory investme...


8624 items for history record.


,title,description,requirements
itemID,,,
I1000011,Front Desk Recruiter/Receptionist,Front Desk Recruiter/Receptionist\nRapidly Gro...,Valid DL\nHigh School Diploma\nAbility to Mult...
I1001363,Food Service Workers - Shedd Aquarium,We are a World Class Aquarium and we connect y...,Friendly. Outgoing. Energetic. Excited about w...
I1001715,Ocean Export Document Processing Agent,The Ocean Export Document Processing Agent wil...,Ability to effectively multi-task\n\nBasic fre...
I1003092,"Amarillo, TX STORE MANAGER CANDIDATE",Are you a take-charge retail manager with a gi...,Ability to read and interpret documents such a...
I1003347,"HR Project Coordinator, NEW GRAD Encouraged to...",Were looking for a recent college graduate who...,1 – 4 years of work experience required prefer...


6405 items for candidates record.
